# Spam Detection Using an LSTM

A deep learning natural language processing project that classifies SMS messages as **spam** or **ham (not spam)** using text preprocessing, tokenization, sequence padding, and a Long Short-Term Memory (LSTM) neural network.

The original assignment prompts have been removed so the notebook focuses on the completed workflow and reproducible code.

## 1. Imports

In [ ]:
import re

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

## 2. Load and Prepare the Dataset

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")

# Keep only the relevant columns
df = df[["v1", "v2"]]
df.columns = ["label", "text"]

# Convert labels to binary values
df["label"] = df["label"].map({"ham": 0, "spam": 1})

df.head()

## 3. Text Preprocessing

The messages are normalized before being converted into numerical sequences.

The preprocessing pipeline:
- Converts text to lowercase
- Removes URLs
- Removes non-alphabetic characters
- Normalizes whitespace

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_text"] = df["text"].apply(clean_text)

X = df["clean_text"]
y = df["label"]

## 4. Training, Validation, and Test Split

In [ ]:
# First split: training set and temporary set
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: validation set and test set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

The original uploaded notebook contained a variable-assignment typo in the second split (`y_train` was assigned instead of `y_val`). It also produced downstream errors because `y_val` was later required by model training. The cleaned portfolio version corrects that assignment so the intended training/validation/test workflow is reproducible.

## 5. Tokenization and Sequence Padding

In [ ]:
vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)
val_sequences = tokenizer.texts_to_sequences(X_val)
test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post"
)

X_val_pad = pad_sequences(
    val_sequences,
    maxlen=max_length,
    padding="post"
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post"
)

## 6. LSTM Model

In [ ]:
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    ),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

## 7. Model Training

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val_pad, y_val)
)

## 8. Test Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

## 9. Predictions and Classification Metrics

In [ ]:
y_pred_probs = model.predict(X_test_pad)
y_pred = (y_pred_probs > 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Summary

This notebook demonstrates an end-to-end deep learning workflow for SMS spam classification:

1. Load and prepare labeled text data.
2. Clean and normalize the messages.
3. Split the data into training, validation, and test sets.
4. Tokenize the training vocabulary.
5. Convert messages into padded numerical sequences.
6. Train an LSTM-based classifier.
7. Evaluate the model on unseen test data.
8. Analyze predictions using a confusion matrix and classification report.

The original notebook's execution errors are not reproduced as portfolio results because they resulted from the variable-assignment issue described above; the cleaned notebook contains the corrected implementation instead.